# Análise exploratória (EDA): temperatura em SP

Objetivo: entender os dados de clima antes de modelar a previsão de temperatura.

- Só **lê** o banco (`dados/warehouse.duckdb`); a carga é feita no `02_open_meteo.ipynb`.
- Usa apenas dado fechado (`fonte = 'archive'`): os dias `forecast` são provisórios.
- Rode com a carga completa; com poucas cidades ou poucos anos, as conclusões enganam.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from clima.db import consultar
from clima.open_meteo import VARS

DB = Path("..") / "dados" / "warehouse.duckdb"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})

# Dado fechado, já com o nome da cidade e as coordenadas.
COM = """
with clima as (
    select c.*, m.nome, m.regiao_imediata, m.lat, m.lng
    from bronze.clima_diario c
    join bronze.ibge_municipios m using (ibge_id)
    where c.fonte = 'archive'
)
"""


def q(sql, params=None):
    """Consulta sobre a visão 'clima' (CTE acima) e devolve um DataFrame."""
    return consultar(COM + sql, DB, params)

## 1. Qualidade dos dados

In [ ]:
display(q("""
    select count(*) as linhas, count(distinct ibge_id) as cidades,
           min(data) as de, max(data) as ate
    from clima
"""))

# Nulos por variável e dias faltando por cidade (buracos > 0 indicam falha na carga).
display(q("select " + ", ".join(f"count(*) - count({v}) as {v}" for v in VARS) + " from clima"))
display(q("""
    select nome, min(data) as de, max(data) as ate, count(*) as dias,
           (max(data) - min(data) + 1) - count(*) as buracos
    from clima group by nome order by buracos desc, nome
""").head(10))

In [ ]:
# Carrega uma vez só; as próximas seções trabalham em memória.
df = q("select ibge_id, nome, regiao_imediata, lat, lng, data, " + ", ".join(VARS) + " from clima")
assert not df.empty, "Sem dados 'archive': rode a carga (notebook 02) antes."

df["data"] = pd.to_datetime(df["data"])
df["ano"], df["mes"], df["dia_do_ano"] = df["data"].dt.year, df["data"].dt.month, df["data"].dt.dayofyear
df["amplitude"] = df["temperature_2m_max"] - df["temperature_2m_min"]   # amplitude térmica do dia
print(df.shape)

## 2. Distribuições

In [ ]:
cols = VARS + ["amplitude"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, c in zip(axes.ravel(), cols):
    ax.hist(df[c].dropna(), bins=50)
    ax.set_title(c, fontsize=9)
fig.tight_layout()
plt.show()

print(f"dias sem chuva: {(df['precipitation_sum'] == 0).mean() * 100:.0f}%")
display(df[cols].describe().T)

## 3. Séries no tempo

In [ ]:
cidade = df["nome"].value_counts().index[0]   # cidade com mais dias; troque pelo nome que quiser
s = df[df["nome"] == cidade].set_index("data")["temperature_2m_mean"]

fig, ax = plt.subplots()
s.plot(ax=ax, alpha=0.4, label="diária")
s.rolling(30, min_periods=15).mean().plot(ax=ax, label="média móvel de 30 dias")
ax.set_title(f"Temperatura média diária: {cidade}")
ax.legend()
plt.show()

In [ ]:
# Média por ano (anos incompletos distorcem: confira a coluna "dias").
anual = df.groupby("ano").agg(temp=("temperature_2m_mean", "mean"), dias=("data", "nunique"))
display(anual)
fig, ax = plt.subplots()
anual["temp"].plot(ax=ax, marker="o", title="Temperatura média por ano")
plt.show()

## 4. Sazonalidade e amplitude térmica

In [ ]:
df.boxplot(column="temperature_2m_mean", by="mes", figsize=(10, 4))
plt.suptitle("")
plt.title("Temperatura média por mês")
plt.show()

fig, ax = plt.subplots()
df.groupby("mes")["amplitude"].mean().plot(kind="bar", ax=ax, title="Amplitude térmica média por mês (°C)")
plt.show()

## 5. Relações entre variáveis

In [ ]:
corr = df[VARS + ["amplitude"]].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.grid(False)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im)
ax.set_title("Correlação entre variáveis")
plt.show()

## 6. Autocorrelação

In [ ]:
serie = s.asfreq("D")   # uma linha por dia (dias faltando viram NaN)
lags = list(range(1, 31))
fig, ax = plt.subplots()
ax.bar(lags, [serie.autocorr(lag=l) for l in lags])
ax.set_title(f"Autocorrelação da temperatura média: {cidade}")
ax.set_xlabel("defasagem (dias)")
plt.show()
print({l: round(float(serie.autocorr(l)), 2) for l in (1, 2, 7, 14, 365)})

## 7. Comparação entre cidades

In [ ]:
por_cidade = df.groupby("nome").agg(temp=("temperature_2m_mean", "mean"), lat=("lat", "first"), lng=("lng", "first"))
fig, ax = plt.subplots()
por_cidade.plot.scatter(x="lat", y="temp", ax=ax, title="Temperatura média por latitude")
plt.show()

pivo = df.pivot_table(index="data", columns="nome", values="temperature_2m_mean")
corr_cidades = pivo.corr()
print("menor correlação entre duas cidades:", round(float(corr_cidades.min().min()), 2))
display(corr_cidades.round(2).iloc[:8, :8])

## 8. Modelos de referência

Antes de qualquer modelo, duas réguas simples que ele precisa vencer:

1. **Climatologia:** prever a média histórica daquele dia do ano (calculada só com o treino).
2. **Persistência:** prever a temperatura de ontem.

O teste usa o último ano de dados, separado por data.

In [ ]:
corte = df["data"].max() - pd.Timedelta(days=365)
treino, teste = df[df["data"] <= corte], df[df["data"] > corte]

# Climatologia: média de cada cidade em cada dia do ano, só com o treino.
clim = treino.groupby(["ibge_id", "dia_do_ano"])["temperature_2m_mean"].mean().rename("prev").reset_index()
av = teste.merge(clim, on=["ibge_id", "dia_do_ano"], how="left")
erro = av["temperature_2m_mean"] - av["prev"]
print(f"Treino até {corte.date()} | teste: {len(teste)} linhas")
print(f"Climatologia: MAE {erro.abs().mean():.2f} °C | RMSE {np.sqrt((erro ** 2).mean()):.2f} °C")

# Persistência: temperatura de ontem (só vale onde existe o dia anterior).
df = df.sort_values(["ibge_id", "data"])
df["ontem"] = df.groupby("ibge_id")["temperature_2m_mean"].shift(1)
df.loc[df.groupby("ibge_id")["data"].diff() != pd.Timedelta(days=1), "ontem"] = np.nan
t = df[df["data"] > corte].dropna(subset=["ontem"])
print(f"Persistência: MAE {(t['temperature_2m_mean'] - t['ontem']).abs().mean():.2f} °C")

In [ ]:
# Anomalia = temperatura menos a média do dia do ano. Mostra a "memória" além da sazonalidade.
df["anomalia"] = df["temperature_2m_mean"] - df.groupby(["ibge_id", "dia_do_ano"])["temperature_2m_mean"].transform("mean")
an = df[df["nome"] == cidade].set_index("data")["anomalia"].asfreq("D")
print({l: round(float(an.autocorr(l)), 2) for l in (1, 2, 3, 7)})

## 9. Conclusões para a modelagem

*(a preencher depois de rodar com a carga completa)*

- Qualidade: o que precisou de tratamento (buracos, nulos, outliers).
- Variáveis candidatas e as que podem ficar de fora.
- Defasagens e médias móveis que valem como variáveis do modelo.
- Quanto o modelo precisa melhorar sobre a climatologia e a persistência para valer a pena.
- Como separar treino e teste (por data, sem embaralhar).